# Acceptance Probability Model

**Input files** (cleaned by `Data_Cleaning.ipynb`):
- `all_info.csv` — full waybill table (coords /1e6, timestamps as datetime, includes `day_of_week` / `is_weekend`)
- `rider.csv` — rider snapshot table (coords /1e6, `dispatch_time` as datetime)
- `courier.csv` — wave table (timestamps as datetime)

## Data Leakage Fixes

| Leak source | Fix |
|-------------|-----|
| `rider_lat/lng` filled with `grab_lat/lng` — grab coords are the outcome of acceptance | Fill missing rider location with median only; never use grab coords |
| `d_rider_to_sender_km` computed from grab coords for grabbed=1 rows | Compute only from rider_snapshot; NaN rows filled with median |
| `grab_lat/lng`, `d_pickup_km`, `d_total_km` left in feature set | Explicitly excluded |
| `hist_grab_rate` shift fails if `dt` dtype is `date` not `Timestamp` | Force `dt` to `datetime64` before groupby |

## 0. Config

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import pickle
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, log_loss, classification_report, f1_score
)

DATA_DIR  = Path('.')         # <- change to your folder path
TEST_DATE = '2022-10-19'      # last day held out as test set
TEST_DT   = pd.Timestamp(TEST_DATE)

# grab_lat / grab_lng are POST-acceptance outcomes -- NEVER include them.
# d_pickup_km and d_total_km are computed from grab coords -- excluded.
# rider_lat / rider_lng come from the dispatch snapshot, NOT grab coords (see Section 5).
# hist_grab_rate is a lagged historical stat (shift=1), NOT a grab coordinate -- it is safe.
FEATURES_ACCEPT = [
    # Distance (pre-acceptance only)
    'd_delivery_km',
    'd_rider_to_sender_km',
    # Time
    'hour_of_day', 'day_of_week', 'is_weekend',
    'is_lunch_peak', 'is_dinner_peak',
    'wait_meal_sec',
    # Rider real-time state (from dispatch snapshot)
    'courier_load',
    'rider_lat', 'rider_lng',
    # Order attributes
    'is_prebook',
    # Historical features (lagged with shift=1 -- no look-ahead)
    'hist_avg_wave_dur', 'hist_avg_wave_cnt',
    'hist_grab_rate',
    'hist_poi_wait',
]

TARGET_ACCEPT = 'is_courier_grabbed'

## 1. Load Cleaned Data

In [2]:
print('Loading data...')

waybill = pd.read_csv(
    DATA_DIR / 'all_info.csv',
    parse_dates=['dt', 'dispatch_time', 'grab_time',
                 'fetch_time', 'arrive_time', 'estimate_meal_prepare_time']
)
print(f'  all_info.csv  : {len(waybill):,} rows')

dispatch_rider = pd.read_csv(DATA_DIR / 'rider.csv', parse_dates=['dispatch_time'])
print(f'  rider.csv     : {len(dispatch_rider):,} rows')

wave = pd.read_csv(
    DATA_DIR / 'courier.csv',
    parse_dates=['dt', 'wave_start_time', 'wave_end_time']
)
print(f'  courier.csv   : {len(wave):,} rows')

# Force dt to datetime64 -- critical for shift() to work correctly later
waybill['dt'] = pd.to_datetime(waybill['dt'])
wave['dt']    = pd.to_datetime(wave['dt'])

print(f"  Positive rate : {waybill['is_courier_grabbed'].mean():.1%}")

Loading data...
  all_info.csv  : 654,343 rows
  rider.csv     : 62,044 rows
  courier.csv   : 206,748 rows
  Positive rate : 86.9%


## 2. Utility

In [3]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorised Haversine distance in kilometres."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return R * 2 * np.arcsin(np.sqrt(a))

## 3. Distance Features

`d_delivery_km` uses only sender/recipient coords -- both known before dispatch, no leakage.  
Do NOT compute `d_pickup_km` or `d_total_km` here -- those require `grab_lat/lng`.

In [4]:
print('Computing distance features...')

waybill_all = waybill.copy()

waybill_all['d_delivery_km'] = haversine_km(
    waybill_all['sender_lat'],    waybill_all['sender_lng'],
    waybill_all['recipient_lat'], waybill_all['recipient_lng']
)
# d_rider_to_sender_km is added in Section 5 from the dispatch snapshot

Computing distance features...


## 4. Time Features

In [5]:
print('Computing time features...')

dt_col = pd.to_datetime(waybill_all['dispatch_time'])

waybill_all['hour_of_day']    = dt_col.dt.hour
waybill_all['day_of_week']    = dt_col.dt.dayofweek
waybill_all['is_weekend']     = (dt_col.dt.dayofweek >= 5).astype(int)
waybill_all['is_lunch_peak']  = dt_col.dt.hour.between(11, 13).astype(int)
waybill_all['is_dinner_peak'] = dt_col.dt.hour.between(17, 20).astype(int)

waybill_all['wait_meal_sec'] = (
    (pd.to_datetime(waybill_all['estimate_meal_prepare_time'])
     - pd.to_datetime(waybill_all['dispatch_time']))
    .dt.total_seconds()
    .clip(lower=0)
)

Computing time features...


## 5. Rider Real-Time Features (from dispatch snapshot)

Key fix: `rider_lat/lng` come from the dispatch snapshot only.  
When no snapshot is found, fill with the global median -- never with `grab_lat/lng`.

In [6]:
print('Joining rider snapshot features...')

if 'current_load' in dispatch_rider.columns and 'courier_load' not in dispatch_rider.columns:
    dispatch_rider = dispatch_rider.rename(columns={'current_load': 'courier_load'})

rider_snapshot = (
    dispatch_rider[['courier_id', 'dispatch_time', 'rider_lat', 'rider_lng', 'courier_load']]
    .drop_duplicates(subset=['courier_id', 'dispatch_time'])
    .sort_values('dispatch_time')
    .reset_index(drop=True)
)

left = (
    waybill_all[['order_id', 'courier_id', 'dispatch_time', 'sender_lat', 'sender_lng']]
    .sort_values('dispatch_time')
    .reset_index(drop=True)
)

matched = pd.merge_asof(
    left,
    rider_snapshot.rename(columns={
        'dispatch_time': 'snap_time',
        'rider_lat':     'snap_rider_lat',
        'rider_lng':     'snap_rider_lng'
    }),
    left_on='dispatch_time',
    right_on='snap_time',
    by='courier_id',
    direction='backward',
    tolerance=pd.Timedelta('10min')
)

# Compute rider->restaurant distance from snapshot position only
has_snap = matched['snap_rider_lat'].notna()
matched['d_rider_to_sender_km'] = np.nan
matched.loc[has_snap, 'd_rider_to_sender_km'] = haversine_km(
    matched.loc[has_snap, 'snap_rider_lat'], matched.loc[has_snap, 'snap_rider_lng'],
    matched.loc[has_snap, 'sender_lat'],     matched.loc[has_snap, 'sender_lng']
)

rider_cols = matched[['order_id', 'snap_rider_lat', 'snap_rider_lng',
                       'courier_load', 'd_rider_to_sender_km']].rename(
    columns={'snap_rider_lat': 'rider_lat', 'snap_rider_lng': 'rider_lng'}
)
waybill_all = waybill_all.merge(rider_cols, on='order_id', how='left')

# Fill missing with MEDIAN only -- never grab coords
for col in ['rider_lat', 'rider_lng', 'd_rider_to_sender_km', 'courier_load']:
    median_val = waybill_all[col].median()
    waybill_all[col] = waybill_all[col].fillna(median_val if pd.notna(median_val) else 0)

waybill_all['courier_load'] = waybill_all['courier_load'].astype(int)

snap_rate = has_snap.mean()
print(f'  Snapshot match rate: {snap_rate:.1%}  '
      f'(remaining {1 - snap_rate:.1%} filled with median)')

Joining rider snapshot features...
  Snapshot match rate: 4.7%  (remaining 95.3% filled with median)


## 6. Rider Historical Features (lag-safe)

`shift(1)` ensures only yesterday-and-before data is used.

In [7]:
print('Computing rider historical features (lag-safe)...')

wave['wave_duration_sec'] = (
    (wave['wave_end_time'] - wave['wave_start_time']).dt.total_seconds()
)

wave_daily = (
    wave.groupby(['dt', 'courier_id'])
    .agg(
        daily_wave_count   = ('wave_id',           'count'),
        daily_avg_wave_dur = ('wave_duration_sec', 'mean'),
    )
    .reset_index()
    .sort_values(['courier_id', 'dt'])
)

wave_daily['hist_avg_wave_dur'] = (
    wave_daily.groupby('courier_id')['daily_avg_wave_dur']
    .transform(lambda x: x.shift(1).expanding().mean())
)
wave_daily['hist_avg_wave_cnt'] = (
    wave_daily.groupby('courier_id')['daily_wave_count']
    .transform(lambda x: x.shift(1).expanding().mean())
)

# hist_grab_rate: lagged daily acceptance rate -- no leakage
grab_rate = (
    waybill.groupby(['dt', 'courier_id'])['is_courier_grabbed']
    .mean()
    .reset_index()
    .rename(columns={'is_courier_grabbed': 'daily_grab_rate'})
    .sort_values(['courier_id', 'dt'])
)
grab_rate['hist_grab_rate'] = (
    grab_rate.groupby('courier_id')['daily_grab_rate']
    .transform(lambda x: x.shift(1).expanding().mean())
)

hist_features = (
    wave_daily[['dt', 'courier_id', 'hist_avg_wave_dur', 'hist_avg_wave_cnt']]
    .merge(grab_rate[['dt', 'courier_id', 'hist_grab_rate']], on=['dt', 'courier_id'], how='outer')
)

waybill_all = waybill_all.merge(hist_features, on=['dt', 'courier_id'], how='left')

Computing rider historical features (lag-safe)...


## 7. Restaurant Historical Wait Time (lag-safe)

In [8]:
print('Computing restaurant historical wait time...')

poi_daily = (
    waybill[waybill['is_courier_grabbed'] == 1]
    .assign(wait_meal_sec=lambda d: (
        (pd.to_datetime(d['estimate_meal_prepare_time']) - pd.to_datetime(d['dispatch_time']))
        .dt.total_seconds().clip(lower=0)
    ))
    .groupby(['dt', 'poi_id'])['wait_meal_sec']
    .mean().reset_index()
    .rename(columns={'wait_meal_sec': 'daily_poi_wait'})
    .sort_values(['poi_id', 'dt'])
)
poi_daily['hist_poi_wait'] = (
    poi_daily.groupby('poi_id')['daily_poi_wait']
    .transform(lambda x: x.shift(1).expanding().mean())
)

waybill_all = waybill_all.merge(
    poi_daily[['dt', 'poi_id', 'hist_poi_wait']], on=['dt', 'poi_id'], how='left'
)

Computing restaurant historical wait time...


## 8. Fill Missing Values

In [9]:
print('Filling missing values...')

HIST_COLS = ['hist_avg_wave_dur', 'hist_avg_wave_cnt', 'hist_grab_rate', 'hist_poi_wait']

for col in HIST_COLS:
    n_missing = waybill_all[col].isna().sum()
    if n_missing > 0:
        med = waybill_all[col].median()
        waybill_all[col] = waybill_all[col].fillna(med if pd.notna(med) else 0)
        print(f'  {col:<28} filled {n_missing:,} rows  (median={med:.2f})')

# Sanity check: only exact leaky column names are blocked.
# hist_grab_rate is a lagged historical stat -- it is safe and intentionally kept.
FORBIDDEN_FEATURES = {'grab_lat', 'grab_lng', 'd_pickup_km', 'd_total_km'}
leaked = [f for f in FEATURES_ACCEPT if f in FORBIDDEN_FEATURES]
assert len(leaked) == 0, f'Leakage detected in feature list: {leaked}'
print('  Leakage check passed -- no grab coords in feature list.')

Filling missing values...
  hist_avg_wave_dur            filled 140,748 rows  (median=2113.03)
  hist_avg_wave_cnt            filled 140,748 rows  (median=7.83)
  hist_grab_rate               filled 136,644 rows  (median=0.88)
  hist_poi_wait                filled 113,577 rows  (median=470.79)
  Leakage check passed -- no grab coords in feature list.


## 9. Build Dataset & Time Split

In [10]:
print('Building acceptance dataset...')

accept_dataset = (
    waybill_all[FEATURES_ACCEPT + [TARGET_ACCEPT, 'order_id', 'courier_id', 'dt']]
    .copy()
    .dropna(subset=['d_delivery_km'])
    .drop_duplicates(subset=['order_id', 'courier_id'])
    .reset_index(drop=True)
)

train_df = accept_dataset[accept_dataset['dt'] <  TEST_DT]
test_df  = accept_dataset[accept_dataset['dt'] == TEST_DT]

X_train = train_df[FEATURES_ACCEPT]
y_train = train_df[TARGET_ACCEPT]
X_test  = test_df[FEATURES_ACCEPT]
y_test  = test_df[TARGET_ACCEPT]

print(f'  Train: {len(train_df):,} rows  |  Test: {len(test_df):,} rows')
print(f'  Train positive rate: {y_train.mean():.1%}')
print(f'  Test  positive rate: {y_test.mean():.1%}')

missing_check = X_train.isnull().mean()
if missing_check.max() > 0:
    print('  WARNING -- features with remaining NaN:')
    print(missing_check[missing_check > 0])
else:
    print('  No missing values in training features.')

Building acceptance dataset...
  Train: 156,832 rows  |  Test: 78,802 rows
  Train positive rate: 86.7%
  Test  positive rate: 87.3%
  No missing values in training features.


## 10. Train Acceptance Probability Model

In [11]:
print('=' * 55)
print('Training Acceptance Probability Model')
print('=' * 55)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f'  Positive: {pos:,}  Negative: {neg:,}  scale_pos_weight: {spw:.3f}')

accept_params = {
    'objective':         'binary',
    'metric':            'auc',
    'learning_rate':     0.05,
    'num_leaves':        63,
    'max_depth':         6,
    'min_child_samples': 100,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.5,
    'reg_lambda':        1.0,
    'scale_pos_weight':  spw,
    'verbose':           -1,
    'n_jobs':            -1,
    'seed':              42,
}

dtrain = lgb.Dataset(X_train, label=y_train)
dval   = lgb.Dataset(X_test,  label=y_test, reference=dtrain)

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100),
]

accept_model = lgb.train(
    params          = accept_params,
    train_set       = dtrain,
    num_boost_round = 1000,
    valid_sets      = [dtrain, dval],
    valid_names     = ['train', 'val'],
    callbacks       = callbacks,
)

Training Acceptance Probability Model
  Positive: 135,925  Negative: 20,907  scale_pos_weight: 0.154
Training until validation scores don't improve for 50 rounds
[100]	train's auc: 0.782363	val's auc: 0.840457
[200]	train's auc: 0.7923	val's auc: 0.841035
Early stopping, best iteration is:
[176]	train's auc: 0.790582	val's auc: 0.841218


## 11. Evaluation

In [12]:
y_prob_test  = accept_model.predict(X_test)
y_prob_train = accept_model.predict(X_train)

# Find best threshold on TRAIN set, apply to TEST set
thresholds = np.linspace(0.1, 0.9, 81)
f1_scores  = [f1_score(y_train, (y_prob_train >= t).astype(int)) for t in thresholds]
best_thr   = thresholds[np.argmax(f1_scores)]
print(f'Best threshold (train F1): {best_thr:.2f}')

y_pred = (y_prob_test >= best_thr).astype(int)

auc  = roc_auc_score(y_test, y_prob_test)
loss = log_loss(y_test, y_prob_test)

print(f'\n--- Test Set Results ---')
print(f'  AUC      : {auc:.4f}')
print(f'  Log Loss : {loss:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Accepted']))

train_auc = roc_auc_score(y_train, y_prob_train)
print(f'Train AUC: {train_auc:.4f}  |  Test AUC: {auc:.4f}  |  Gap: {train_auc - auc:.4f}')
if train_auc - auc > 0.05:
    print('  WARNING: large train/test gap -- consider stronger regularisation.')

Best threshold (train F1): 0.15

--- Test Set Results ---
  AUC      : 0.8412
  Log Loss : 0.4953

Classification Report:
              precision    recall  f1-score   support

    Rejected       0.58      0.09      0.16     10002
    Accepted       0.88      0.99      0.93     68800

    accuracy                           0.88     78802
   macro avg       0.73      0.54      0.55     78802
weighted avg       0.84      0.88      0.84     78802

Train AUC: 0.7906  |  Test AUC: 0.8412  |  Gap: -0.0506


## 12. Feature Importance

In [13]:
importance = pd.DataFrame({
    'feature':    FEATURES_ACCEPT,
    'importance': accept_model.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)

print('Top 15 Features (by gain):')
print(importance.head(15).to_string(index=False))

top5 = importance.head(5)['feature'].tolist()
print(f'\nTop 5: {top5}')

Top 15 Features (by gain):
             feature   importance
      hist_grab_rate 48093.301506
         hour_of_day 13257.368717
       d_delivery_km  8646.013480
       wait_meal_sec  8106.916111
   hist_avg_wave_cnt  7339.464610
   hist_avg_wave_dur  4283.322352
       hist_poi_wait  4161.226994
           rider_lat  1890.550252
         day_of_week  1266.715485
          is_prebook  1091.714421
           rider_lng   970.058608
        courier_load   733.489716
d_rider_to_sender_km   710.256935
       is_lunch_peak    19.461976
      is_dinner_peak    12.508720

Top 5: ['hist_grab_rate', 'hour_of_day', 'd_delivery_km', 'wait_meal_sec', 'hist_avg_wave_cnt']


## 13. Save Model

In [14]:
print('Saving model...')

accept_model.save_model(str(DATA_DIR / 'accept_model.txt'))

with open(DATA_DIR / 'accept_model_meta.pkl', 'wb') as f:
    pickle.dump({
        'features_accept': FEATURES_ACCEPT,
        'best_threshold':  best_thr,
        'test_auc':        auc,
    }, f)

print('  Saved: accept_model.txt')
print('  Saved: accept_model_meta.pkl')
print('Done.')

Saving model...
  Saved: accept_model.txt
  Saved: accept_model_meta.pkl
Done.
